# Interval Decoding

This notebook contains plotting only. Both decoders save result tables, and all visualization now happens here.


In [25]:
from pathlib import Path
import json
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [26]:
import re

TARGET_ORDER = ['Cue', 'Outcome', 'R1 choice', 'R2 choice']

BIN_RUN_PREFIXES = [
    'run_logreg',
]
SESSION_RUN_PREFIXES = [
    'run_session_logreg',
]
ASSEMBLIES = None  # None -> auto-discover all Assembly### runs, or set e.g. ['Assembly012']
BALANCED_ACCURACY_INTERVALS = [
    'cue_entry_interval',
    'R1_entry_interval',
    'R2_entry_interval',
]  # Set to None to show all intervals in balanced-accuracy plots.

print('Train command examples:')
print('python scripted_plotting/animal_6_analysis/interval_decoding_train.py --run-name run_logreg --assembly-col all --n-jobs 4')
print('python scripted_plotting/animal_6_analysis/interval_decoding_train.py --run-name run_logreg --assembly-col all --n-jobs 4 --bootstrap-n 1000 --shuffle-n 500')
print('python scripted_plotting/animal_6_analysis/interval_session_decoding_train.py --run-name run_session_logreg --assembly-col all --n-jobs 4 --bootstrap-n 1000 --shuffle-n 500')


def _find_runs_base(candidate_rels) -> Path:
    cwd = Path.cwd().resolve()
    for root in [cwd, *cwd.parents]:
        for rel in candidate_rels:
            cand = root / rel
            if cand.exists():
                return cand
    raise FileNotFoundError(f'Could not locate any run directory from {candidate_rels}')


def _read_table(run_dir: Path, stem: str) -> pd.DataFrame:
    parquet_path = run_dir / f'{stem}.parquet'
    csv_path = run_dir / f'{stem}.csv'
    if parquet_path.exists():
        try:
            return pd.read_parquet(parquet_path)
        except Exception:
            if not csv_path.exists():
                raise
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'Missing {stem} table in {run_dir}')


def _assembly_sort_key(name: str):
    match = re.fullmatch(r'Assembly(\d+)', str(name))
    if match:
        return (0, int(match.group(1)))
    return (1, str(name))


def _discover_assembly_names(runs_base: Path, run_prefixes) -> list[str]:
    assemblies = set()
    for prefix in run_prefixes:
        pattern = re.compile(re.escape(prefix) + r'__(Assembly\d+)$')
        for child in runs_base.iterdir():
            if not child.is_dir():
                continue
            match = pattern.fullmatch(child.name)
            if match:
                assemblies.add(match.group(1))
    return sorted(assemblies, key=_assembly_sort_key)


def build_run_names(run_prefixes, candidate_rels, assemblies=None):
    runs_base = _find_runs_base(candidate_rels)
    selected_assemblies = list(assemblies) if assemblies else _discover_assembly_names(runs_base, run_prefixes)
    run_names = []
    for prefix in run_prefixes:
        if selected_assemblies:
            run_names.extend(
                [
                    f'{prefix}__{assembly}'
                    for assembly in selected_assemblies
                    if (runs_base / f'{prefix}__{assembly}').exists()
                ]
            )
        elif (runs_base / prefix).exists():
            run_names.append(prefix)
    return run_names


def _bh_fdr(p_values) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    q_values = np.full(p_values.shape, np.nan, dtype=float)
    valid = np.isfinite(p_values)
    if not valid.any():
        return q_values

    p_valid = p_values[valid]
    order = np.argsort(p_valid)
    ranked = p_valid[order]
    scale = ranked.size / np.arange(1, ranked.size + 1, dtype=float)
    q_ranked = np.minimum.accumulate((ranked * scale)[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0.0, 1.0)

    q_valid = np.empty_like(p_valid)
    q_valid[order] = q_ranked
    q_values[valid] = q_valid
    return q_values


def _add_shuffle_fdr_columns(results_df: pd.DataFrame) -> pd.DataFrame:
    out = results_df.copy()
    if 'p_shuffle' not in out.columns:
        out['p_shuffle_fdr_bh'] = np.nan
        out['significant_shuffle_fdr_bh_0_05'] = np.nan
        return out

    q_values = _bh_fdr(pd.to_numeric(out['p_shuffle'], errors='coerce').to_numpy(dtype=float))
    out['p_shuffle_fdr_bh'] = q_values

    significant = pd.Series(np.nan, index=out.index, dtype=object)
    valid = np.isfinite(q_values)
    significant.loc[valid] = q_values[valid] < 0.05
    out['significant_shuffle_fdr_bh_0_05'] = significant
    return out


def load_run_bundle(run_names, candidate_rels):
    empty = {
        'base_dir': None,
        'run_dirs': [],
        'results_df': pd.DataFrame(),
        'diagnostics_df': pd.DataFrame(),
        'meta_df': pd.DataFrame(),
        'config_df': pd.DataFrame(),
    }
    if not run_names:
        return empty

    runs_base = _find_runs_base(candidate_rels)
    run_dirs = [runs_base / name for name in run_names]

    results_parts = []
    diagnostics_parts = []
    meta_by_run = {}
    config_by_run = {}
    for run_dir in run_dirs:
        results_df_run = _read_table(run_dir, 'results')
        diagnostics_df_run = _read_table(run_dir, 'diagnostics')
        results_df_run['run_name'] = run_dir.name
        results_df_run = _add_shuffle_fdr_columns(results_df_run)
        diagnostics_df_run['run_name'] = run_dir.name
        results_parts.append(results_df_run)
        diagnostics_parts.append(diagnostics_df_run)
        meta_by_run[run_dir.name] = json.loads((run_dir / 'meta.json').read_text())
        config_by_run[run_dir.name] = json.loads((run_dir / 'config.json').read_text())

    return {
        'base_dir': runs_base,
        'run_dirs': run_dirs,
        'results_df': pd.concat(results_parts, ignore_index=True),
        'diagnostics_df': pd.concat(diagnostics_parts, ignore_index=True),
        'meta_df': pd.DataFrame(meta_by_run).T,
        'config_df': pd.DataFrame(config_by_run).T,
    }


def _target_order(values):
    present = [target for target in TARGET_ORDER if target in set(values)]
    present += sorted(set(values) - set(present))
    return present


def _resolve_interval_order(values, interval_names=None):
    present = list(dict.fromkeys(pd.Series(values).astype(str)))
    if interval_names is None:
        return present
    requested = [str(interval_name) for interval_name in interval_names]
    present_set = set(present)
    return [interval_name for interval_name in requested if interval_name in present_set]


def _filter_interval_rows(df: pd.DataFrame, interval_names=None):
    interval_order = _resolve_interval_order(df['interval_name'], interval_names)
    if interval_names is None:
        return df.copy(), interval_order
    if not interval_order:
        return df.iloc[0:0].copy(), interval_order
    filtered = df[df['interval_name'].astype(str).isin(interval_order)].copy()
    return filtered, interval_order


def _parse_session_date_token(session_name: str):
    for token in str(session_name).split('_'):
        try:
            return datetime.strptime(token, '%Y-%m-%d')
        except ValueError:
            continue
    return None


def _session_sort_key(session_name: str):
    parsed = _parse_session_date_token(session_name)
    fallback = datetime.max if parsed is None else parsed
    return fallback, str(session_name)


def _preview_runs(run_names):
    if len(run_names) <= 6:
        return run_names
    return [*run_names[:3], '...', *run_names[-3:]]


BIN_RUN_NAMES = build_run_names(
    BIN_RUN_PREFIXES,
    [
        'scripted_plotting/animal_6_analysis/interval_decoding_runs',
        'interval_decoding_runs',
    ],
    assemblies=ASSEMBLIES,
)
SESSION_RUN_NAMES = build_run_names(
    SESSION_RUN_PREFIXES,
    [
        'scripted_plotting/animal_6_analysis/interval_session_decoding_runs',
        'interval_session_decoding_runs',
    ],
    assemblies=ASSEMBLIES,
)

print(f'Configured bin runs ({len(BIN_RUN_NAMES)}):', _preview_runs(BIN_RUN_NAMES))
print(f'Configured session runs ({len(SESSION_RUN_NAMES)}):', _preview_runs(SESSION_RUN_NAMES))



Train command examples:
python scripted_plotting/animal_6_analysis/interval_decoding_train.py --run-name run_logreg --assembly-col all --n-jobs 4
python scripted_plotting/animal_6_analysis/interval_decoding_train.py --run-name run_logreg --assembly-col all --n-jobs 4 --bootstrap-n 1000 --shuffle-n 500
python scripted_plotting/animal_6_analysis/interval_session_decoding_train.py --run-name run_session_logreg --assembly-col all --n-jobs 4 --bootstrap-n 1000 --shuffle-n 500
Configured bin runs (23): ['run_logreg__Assembly001', 'run_logreg__Assembly002', 'run_logreg__Assembly003', '...', 'run_logreg__Assembly021', 'run_logreg__Assembly022', 'run_logreg__Assembly023']
Configured session runs (23): ['run_session_logreg__Assembly001', 'run_session_logreg__Assembly002', 'run_session_logreg__Assembly003', '...', 'run_session_logreg__Assembly021', 'run_session_logreg__Assembly022', 'run_session_logreg__Assembly023']


## Bin-Wise Decoder Runs


In [27]:
bin_bundle = load_run_bundle(
    BIN_RUN_NAMES,
    [
        'scripted_plotting/animal_6_analysis/interval_decoding_runs',
        'interval_decoding_runs',
    ],
)
bin_results_df = bin_bundle['results_df']
bin_diagnostics_df = bin_bundle['diagnostics_df']

if bin_results_df.empty:
    print('No bin-wise runs loaded.')
else:
    print(f"Loaded bin-wise runs: {[run_dir.name for run_dir in bin_bundle['run_dirs']]}")
    display(bin_bundle['meta_df'])
    display(bin_bundle['config_df'])
    display(bin_results_df.head())
    display(bin_diagnostics_df.head())


Loaded bin-wise runs: ['run_logreg__Assembly001', 'run_logreg__Assembly002', 'run_logreg__Assembly003', 'run_logreg__Assembly004', 'run_logreg__Assembly005', 'run_logreg__Assembly006', 'run_logreg__Assembly007', 'run_logreg__Assembly008', 'run_logreg__Assembly009', 'run_logreg__Assembly010', 'run_logreg__Assembly011', 'run_logreg__Assembly012', 'run_logreg__Assembly013', 'run_logreg__Assembly014', 'run_logreg__Assembly015', 'run_logreg__Assembly016', 'run_logreg__Assembly017', 'run_logreg__Assembly018', 'run_logreg__Assembly019', 'run_logreg__Assembly020', 'run_logreg__Assembly021', 'run_logreg__Assembly022', 'run_logreg__Assembly023']


,feature_cols,n_features,session_col,trial_col,interval_col,bin_width_s,elapsed_s,task_count,target_filter,interval_filter,...,bootstrap_metric,bootstrap_ci,bootstrap_within_group,min_session_date,drop_unparseable_session_dates,assembly,session_date_filter,sessions_requested,sessions_after_date_filter,sessions_dropped_by_date_filter
run_logreg__Assembly001,[Assembly001],1,session_id,trial_id,interval_name,0.04,5956.17261,1320,None,None,...,balanced_accuracy,0.95,True,2024-11-27,True,Assembly001,2024-11-27,31,23,8
run_logreg__Assembly002,[Assembly002],1,session_id,trial_id,interval_name,0.04,5916.517875,1320,None,None,...,balanced_accuracy,0.95,True,2024-11-27,True,Assembly002,2024-11-27,31,23,8
run_logreg__Assembly003,[Assembly003],1,session_id,trial_id,interval_name,0.04,5907.774159,1320,None,None,...,balanced_accuracy,0.95,True,2024-11-27,True,Assembly003,2024-11-27,31,23,8
run_logreg__Assembly004,[Assembly004],1,session_id,trial_id,interval_name,0.04,5477.298396,1320,None,None,...,balanced_accuracy,0.95,True,2024-11-27,True,Assembly004,2024-11-27,31,23,8
run_logreg__Assembly005,[Assembly005],1,session_id,trial_id,interval_name,0.04,5528.010223,1320,None,None,...,balanced_accuracy,0.95,True,2024-11-27,True,Assembly005,2024-11-27,31,23,8
run_logreg__Assembly006,[Assembly006],1,session_id,trial_id,interval_name,0.04,5920.844473,1320,None,None,...,balanced_accuracy,0.95,True,2024-11-27,True,Assembly006,2024-11-27,31,23,8
run_logreg__Assembly007,[Assembly007],1,session_id,trial_id,interval_name,0.04,5515.67138,1320,None,None,...,balanced_accuracy,0.95,True,2024-11-27,True,Assembly007,2024-11-27,31,23,8
run_logreg__Assembly008,[Assembly008],1,session_id,trial_id,interval_name,0.04,5474.681247,1320,None,None,...,balanced_accuracy,0.95,True,2024-11-27,True,Assembly008,2024-11-27,31,23,8
run_logreg__Assembly009,[Assembly009],1,session_id,trial_id,interval_name,0.04,5563.829189,1320,None,None,...,balanced_accuracy,0.95,True,2024-11-27,True,Assembly009,2024-11-27,31,23,8
run_logreg__Assembly010,[Assembly010],1,session_id,trial_id,interval_name,0.04,5608.720006,1320,None,None,...,balanced_accuracy,0.95,True,2024-11-27,True,Assembly010,2024-11-27,31,23,8


,assembly_col,window_half_width,min_samples_per_time,min_class_count,max_cv_splits,random_state,shuffle_n,bootstrap_n,bootstrap_ci,use_group_cv,min_session_date,drop_unparseable_session_dates,excluded_intervals,shuffle_within_group,bootstrap_within_group
run_logreg__Assembly001,Assembly001,1,20,4,5,42,500,1000,0.95,True,2024-11-27,True,None,True,True
run_logreg__Assembly002,Assembly002,1,20,4,5,42,500,1000,0.95,True,2024-11-27,True,None,True,True
run_logreg__Assembly003,Assembly003,1,20,4,5,42,500,1000,0.95,True,2024-11-27,True,None,True,True
run_logreg__Assembly004,Assembly004,1,20,4,5,42,500,1000,0.95,True,2024-11-27,True,None,True,True
run_logreg__Assembly005,Assembly005,1,20,4,5,42,500,1000,0.95,True,2024-11-27,True,None,True,True
run_logreg__Assembly006,Assembly006,1,20,4,5,42,500,1000,0.95,True,2024-11-27,True,None,True,True
run_logreg__Assembly007,Assembly007,1,20,4,5,42,500,1000,0.95,True,2024-11-27,True,None,True,True
run_logreg__Assembly008,Assembly008,1,20,4,5,42,500,1000,0.95,True,2024-11-27,True,None,True,True
run_logreg__Assembly009,Assembly009,1,20,4,5,42,500,1000,0.95,True,2024-11-27,True,None,True,True
run_logreg__Assembly010,Assembly010,1,20,4,5,42,500,1000,0.95,True,2024-11-27,True,None,True,True


,backend,assembly,n_features,target_name,target_col,interval_name,model_bin,time_rel_bin,time_bin_idx,time_rel,...,bootstrap_ci_high,bootstrap_effect,p_bootstrap_above_chance,significant_bootstrap_0_05,accuracy,balanced_accuracy,n_splits,cv_strategy,fit_time_s,run_name
0,logreg,Assembly001,1,Outcome,outcome_binary,R1_entry_interval,0,0,0.5,0.012821,...,0.516083,0.001637,0.770230,False,0.492469,0.492806,5,stratified_group,0.001540,run_logreg__Assembly001
1,logreg,Assembly001,1,Outcome,outcome_binary,R1_entry_interval,1,1,1.0,0.025641,...,0.524720,-0.004290,0.571429,False,0.497242,0.492936,5,stratified_group,0.001408,run_logreg__Assembly001
2,logreg,Assembly001,1,Outcome,outcome_binary,R1_entry_interval,2,2,2.0,0.051282,...,0.530377,-0.005416,0.310689,False,0.505568,0.500783,5,stratified_group,0.001419,run_logreg__Assembly001
3,logreg,Assembly001,1,Outcome,outcome_binary,R1_entry_interval,3,3,3.0,0.076923,...,0.552079,-0.006007,0.054945,False,0.523363,0.516575,5,stratified_group,0.001391,run_logreg__Assembly001
4,logreg,Assembly001,1,Outcome,outcome_binary,R1_entry_interval,4,4,4.0,0.102564,...,0.532284,-0.005187,0.371628,False,0.504683,0.500159,5,stratified_group,0.001391,run_logreg__Assembly001


,target_name,target_col,interval_name,model_bin,time_rel_bin,time_bin_idx,time_rel,n_samples,n_sessions,n_classes,min_class_count,run_name
0,Outcome,outcome_binary,R1_entry_interval,0,0,0.5,0.012821,2318,20,2,1153,run_logreg__Assembly001
1,Outcome,outcome_binary,R1_entry_interval,1,1,1.0,0.025641,2318,20,2,1153,run_logreg__Assembly001
2,Outcome,outcome_binary,R1_entry_interval,2,2,2.0,0.051282,2318,20,2,1153,run_logreg__Assembly001
3,Outcome,outcome_binary,R1_entry_interval,3,3,3.0,0.076923,2318,20,2,1153,run_logreg__Assembly001
4,Outcome,outcome_binary,R1_entry_interval,4,4,4.0,0.102564,2318,20,2,1153,run_logreg__Assembly001


In [28]:
def plot_metric_lines(results_df: pd.DataFrame, metric: str = 'balanced_accuracy', interval_names=None) -> None:
    valid = results_df[results_df[metric].notna()].copy()
    if valid.empty:
        print(f'No rows with metric={metric}.')
        return

    for run_name, run_df in valid.groupby('run_name', sort=False):
        run_df, run_interval_order = _filter_interval_rows(run_df, interval_names)
        if run_df.empty:
            continue
        for target_name in _target_order(run_df['target_name'].astype(str)):
            target_df = run_df[run_df['target_name'].astype(str) == target_name].copy()
            target_df, target_interval_order = _filter_interval_rows(target_df, run_interval_order)
            if target_df.empty:
                continue
            interval_order_map = {name: idx for idx, name in enumerate(target_interval_order)}
            plot_df = (
                target_df.assign(
                    interval_name=target_df['interval_name'].astype(str),
                    _interval_order=target_df['interval_name'].astype(str).map(interval_order_map),
                )
                .sort_values(['_interval_order', 'model_bin'])
                .drop(columns='_interval_order')
            )
            fig = px.line(
                plot_df,
                x='model_bin',
                y=metric,
                color='interval_name',
                markers=True,
                title=f'{metric} | run={run_name} | target={target_name}',
                hover_data=[
                    col for col in [
                        'n_samples', 'n_splits', 'fit_time_s', 'p_above_chance',
                        'p_shuffle', 'bootstrap_ci_low', 'bootstrap_ci_high'
                    ] if col in target_df.columns
                ],
                category_orders={'interval_name': target_interval_order},
            )
            fig.update_layout(template='plotly_white')
            fig.show()


def _build_target_heatmap(df: pd.DataFrame, target_name: str, value_col: str, extra_cols):
    sub = df[df['target_name'].astype(str) == str(target_name)].copy()
    intervals = list(dict.fromkeys(sub['interval_name'].astype(str)))
    bins = sorted(pd.to_numeric(sub['model_bin'], errors='coerce').dropna().astype(int).unique().tolist())
    z = np.full((len(intervals), len(bins)), np.nan, dtype=float)
    custom = np.full((len(intervals), len(bins), len(extra_cols)), np.nan, dtype=float)

    interval_to_i = {name: i for i, name in enumerate(intervals)}
    bin_to_j = {int(model_bin): j for j, model_bin in enumerate(bins)}
    for rec in sub.itertuples(index=False):
        i = interval_to_i[str(rec.interval_name)]
        j = bin_to_j[int(rec.model_bin)]
        value = getattr(rec, value_col)
        z[i, j] = float(value) if pd.notna(value) else np.nan
        for extra_idx, col in enumerate(extra_cols):
            if hasattr(rec, col):
                raw = getattr(rec, col)
                custom[i, j, extra_idx] = float(raw) if pd.notna(raw) else np.nan
    return intervals, bins, z, custom


def plot_metric_heatmaps(
    results_df: pd.DataFrame,
    value_col: str,
    title_prefix: str,
    extra_cols,
    colorscale: str = 'Viridis',
    zmid=None,
) -> None:
    valid = results_df[results_df[value_col].notna()].copy()
    if valid.empty:
        print(f'No rows with {value_col}.')
        return

    for run_name, run_df in valid.groupby('run_name', sort=False):
        for target_name in _target_order(run_df['target_name'].astype(str)):
            intervals, bins, z, custom = _build_target_heatmap(run_df, target_name, value_col, extra_cols)
            if not intervals or not bins:
                continue
            hover_lines = [f'{value_col}=%{{z:.3f}}']
            for extra_idx, col in enumerate(extra_cols):
                hover_lines.append(f'{col}=%{{customdata[{extra_idx}]:.3f}}')
            hovertemplate = 'target=' + target_name + '<br>interval=%{y}<br>model_bin=%{x}<br>' + '<br>'.join(hover_lines) + '<extra></extra>'
            fig = go.Figure(
                data=go.Heatmap(
                    x=bins,
                    y=intervals,
                    z=z,
                    customdata=custom,
                    colorscale=colorscale,
                    zmid=zmid,
                    colorbar={'title': value_col},
                    hovertemplate=hovertemplate,
                )
            )
            fig.update_layout(
                template='plotly_white',
                title=f'{title_prefix} | run={run_name} | target={target_name}',
                xaxis_title='model_bin',
                yaxis_title='interval',
                height=max(360, 55 * len(intervals)),
            )
            fig.show()


### CV Performance


In [29]:
# plot_metric_lines(
#     bin_results_df,
#     metric='balanced_accuracy',
#     interval_names=BALANCED_ACCURACY_INTERVALS,
# )
# plot_metric_lines(bin_results_df, metric='accuracy')


### Bootstrap Summary


In [30]:
# plot_metric_heatmaps(
#     results_df=bin_results_df,
#     value_col='bootstrap_mean',
#     title_prefix='Bootstrap mean balanced accuracy',
#     extra_cols=['bootstrap_ci_low', 'bootstrap_ci_high', 'p_bootstrap_above_chance'],
#     colorscale='Viridis',
#     zmid=None,
# )


### Shuffle Control


In [31]:
# plot_metric_heatmaps(
#     results_df=bin_results_df,
#     value_col='shuffle_effect',
#     title_prefix='Shuffle effect',
#     extra_cols=['shuffle_null_mean', 'p_shuffle'],
#     colorscale='RdBu_r',
#     zmid=0.0,
# )


### Best Bins


In [32]:
bin_summary_df = (
    bin_results_df[bin_results_df['balanced_accuracy'].notna()]
    .sort_values(
        ['run_name', 'target_name', 'interval_name', 'balanced_accuracy', 'bootstrap_mean'],
        ascending=[True, True, True, False, False],
        na_position='last',
    )
    .groupby(['run_name', 'target_name', 'interval_name'], as_index=False)
    .first()
)

bin_summary_cols = [
    'run_name', 'target_name', 'interval_name', 'model_bin', 'balanced_accuracy',
    'accuracy', 'p_above_chance', 'p_shuffle', 'bootstrap_ci_low', 'bootstrap_ci_high'
]
if bin_summary_df.empty:
    print('No bin-wise summary rows available.')
else:
    display(bin_summary_df[bin_summary_cols].sort_values(['run_name', 'target_name', 'interval_name']))


,run_name,target_name,interval_name,model_bin,balanced_accuracy,accuracy,p_above_chance,p_shuffle,bootstrap_ci_low,bootstrap_ci_high
0,run_logreg__Assembly001,Cue,R1_entry_interval,17,0.523352,0.524603,1.459600e-02,0.003992,0.505278,0.542260
1,run_logreg__Assembly001,Cue,R1_exit_interval,2,0.520424,0.518974,7.014754e-02,0.009980,0.502882,0.529095
2,run_logreg__Assembly001,Cue,R2_entry_interval,26,0.518360,0.517396,6.473014e-02,0.035928,0.499837,0.532943
3,run_logreg__Assembly001,Cue,R2_exit_interval,23,0.517884,0.517548,6.735471e-02,0.041916,0.499952,0.533841
4,run_logreg__Assembly001,Cue,cue_entry_interval,23,0.520049,0.521508,1.056912e-02,0.015968,0.508574,0.540160
...,...,...,...,...,...,...,...,...,...,...
777,run_logreg__Assembly023,R2 choice,cue_entry_interval,28,0.590835,0.565422,7.127442e-09,0.029940,0.499218,0.620766
778,run_logreg__Assembly023,R2 choice,cue_exit_interval,24,0.601647,0.590712,1.637578e-16,0.003992,0.538288,0.632588
779,run_logreg__Assembly023,R2 choice,nextto_cue_interval,3,0.570883,0.550497,2.140614e-05,0.568862,0.480131,0.609918
780,run_logreg__Assembly023,R2 choice,pre_cue_interval,3,0.555734,0.524923,4.235961e-02,0.534930,0.468273,0.573268


## Session-Level Decoder Runs

Each decoder is fit separately inside each session using all bins from an interval as one per-trial feature vector.


In [33]:
session_bundle = load_run_bundle(
    SESSION_RUN_NAMES,
    [
        'scripted_plotting/animal_6_analysis/interval_session_decoding_runs',
        'interval_session_decoding_runs',
    ],
)
session_results_df = session_bundle['results_df']
session_diagnostics_df = session_bundle['diagnostics_df']

if session_results_df.empty:
    print('No session-level runs loaded.')
else:
    print(f"Loaded session-level runs: {[run_dir.name for run_dir in session_bundle['run_dirs']]}")
    display(session_bundle['meta_df'])
    display(session_bundle['config_df'])
    display(session_results_df.head())
    display(session_diagnostics_df.head())


Loaded session-level runs: ['run_session_logreg__Assembly001', 'run_session_logreg__Assembly002', 'run_session_logreg__Assembly003', 'run_session_logreg__Assembly004', 'run_session_logreg__Assembly005', 'run_session_logreg__Assembly006', 'run_session_logreg__Assembly007', 'run_session_logreg__Assembly008', 'run_session_logreg__Assembly009', 'run_session_logreg__Assembly010', 'run_session_logreg__Assembly011', 'run_session_logreg__Assembly012', 'run_session_logreg__Assembly013', 'run_session_logreg__Assembly014', 'run_session_logreg__Assembly015', 'run_session_logreg__Assembly016', 'run_session_logreg__Assembly017', 'run_session_logreg__Assembly018', 'run_session_logreg__Assembly019', 'run_session_logreg__Assembly020', 'run_session_logreg__Assembly021', 'run_session_logreg__Assembly022', 'run_session_logreg__Assembly023']


,decoder_scope,feature_cols,feature_vector_type,n_features,session_col,trial_col,interval_col,elapsed_s,task_count,target_filter,...,bootstrap_n,bootstrap_metric,bootstrap_ci,min_session_date,drop_unparseable_session_dates,assembly,session_date_filter,sessions_requested,sessions_after_date_filter,sessions_dropped_by_date_filter
run_session_logreg__Assembly001,session_interval,[Assembly001],interval_bin_stack,40,session_id,trial_id,interval_name,5036.217699,792,None,...,1000,balanced_accuracy,0.95,2024-11-27,True,Assembly001,2024-11-27,31,23,8
run_session_logreg__Assembly002,session_interval,[Assembly002],interval_bin_stack,40,session_id,trial_id,interval_name,5406.260832,792,None,...,1000,balanced_accuracy,0.95,2024-11-27,True,Assembly002,2024-11-27,31,23,8
run_session_logreg__Assembly003,session_interval,[Assembly003],interval_bin_stack,40,session_id,trial_id,interval_name,4990.471213,792,None,...,1000,balanced_accuracy,0.95,2024-11-27,True,Assembly003,2024-11-27,31,23,8
run_session_logreg__Assembly004,session_interval,[Assembly004],interval_bin_stack,40,session_id,trial_id,interval_name,4943.553354,792,None,...,1000,balanced_accuracy,0.95,2024-11-27,True,Assembly004,2024-11-27,31,23,8
run_session_logreg__Assembly005,session_interval,[Assembly005],interval_bin_stack,40,session_id,trial_id,interval_name,5474.434828,792,None,...,1000,balanced_accuracy,0.95,2024-11-27,True,Assembly005,2024-11-27,31,23,8
run_session_logreg__Assembly006,session_interval,[Assembly006],interval_bin_stack,40,session_id,trial_id,interval_name,5139.720139,792,None,...,1000,balanced_accuracy,0.95,2024-11-27,True,Assembly006,2024-11-27,31,23,8
run_session_logreg__Assembly007,session_interval,[Assembly007],interval_bin_stack,40,session_id,trial_id,interval_name,5130.167023,792,None,...,1000,balanced_accuracy,0.95,2024-11-27,True,Assembly007,2024-11-27,31,23,8
run_session_logreg__Assembly008,session_interval,[Assembly008],interval_bin_stack,40,session_id,trial_id,interval_name,5423.275592,792,None,...,1000,balanced_accuracy,0.95,2024-11-27,True,Assembly008,2024-11-27,31,23,8
run_session_logreg__Assembly009,session_interval,[Assembly009],interval_bin_stack,40,session_id,trial_id,interval_name,5376.190921,792,None,...,1000,balanced_accuracy,0.95,2024-11-27,True,Assembly009,2024-11-27,31,23,8
run_session_logreg__Assembly010,session_interval,[Assembly010],interval_bin_stack,40,session_id,trial_id,interval_name,5067.010869,792,None,...,1000,balanced_accuracy,0.95,2024-11-27,True,Assembly010,2024-11-27,31,23,8


,assembly_col,min_trials_per_session,min_class_count,max_cv_splits,random_state,shuffle_n,bootstrap_n,bootstrap_ci,min_session_date,drop_unparseable_session_dates,excluded_intervals
run_session_logreg__Assembly001,Assembly001,20,4,5,42,1000,1000,0.95,2024-11-27,True,None
run_session_logreg__Assembly002,Assembly002,20,4,5,42,1000,1000,0.95,2024-11-27,True,None
run_session_logreg__Assembly003,Assembly003,20,4,5,42,1000,1000,0.95,2024-11-27,True,None
run_session_logreg__Assembly004,Assembly004,20,4,5,42,1000,1000,0.95,2024-11-27,True,None
run_session_logreg__Assembly005,Assembly005,20,4,5,42,1000,1000,0.95,2024-11-27,True,None
run_session_logreg__Assembly006,Assembly006,20,4,5,42,1000,1000,0.95,2024-11-27,True,None
run_session_logreg__Assembly007,Assembly007,20,4,5,42,1000,1000,0.95,2024-11-27,True,None
run_session_logreg__Assembly008,Assembly008,20,4,5,42,1000,1000,0.95,2024-11-27,True,None
run_session_logreg__Assembly009,Assembly009,20,4,5,42,1000,1000,0.95,2024-11-27,True,None
run_session_logreg__Assembly010,Assembly010,20,4,5,42,1000,1000,0.95,2024-11-27,True,None


,backend,decoder_scope,assembly,session_id,target_name,target_col,interval_name,n_trials,n_classes,min_class_count,...,bootstrap_ci_high,bootstrap_effect,p_bootstrap_above_chance,significant_bootstrap_0_05,accuracy,balanced_accuracy,n_splits,cv_strategy,fit_time_s,run_name
0,logreg,session_interval,Assembly001,2024-11-28_17-41,Outcome,outcome_binary,R1_entry_interval,133,2,59,...,0.571381,0.000174,0.612388,False,0.488604,0.487879,5,stratified,0.001596,run_session_logreg__Assembly001
1,logreg,session_interval,Assembly001,2024-12-02_16-09,Outcome,outcome_binary,R1_entry_interval,105,2,45,...,0.538898,-0.001682,0.895105,False,0.438095,0.438889,5,stratified,0.001351,run_session_logreg__Assembly001
2,logreg,session_interval,Assembly001,2024-12-03_16-23,Outcome,outcome_binary,R1_entry_interval,104,2,47,...,0.660728,-0.002288,0.117882,False,0.556667,0.559495,5,stratified,0.001330,run_session_logreg__Assembly001
3,logreg,session_interval,Assembly001,2024-12-04_18-06,Outcome,outcome_binary,R1_entry_interval,120,2,55,...,0.550431,-0.000063,0.770230,False,0.466667,0.465734,5,stratified,0.001384,run_session_logreg__Assembly001
4,logreg,session_interval,Assembly001,2024-12-06_16-49,Outcome,outcome_binary,R1_entry_interval,95,2,43,...,0.617465,0.003294,0.377622,False,0.515789,0.519394,5,stratified,0.001286,run_session_logreg__Assembly001


,session_id,target_name,target_col,interval_name,n_trials,n_classes,min_class_count,n_bins_used,run_name
0,2024-11-28_17-41,Outcome,outcome_binary,R1_entry_interval,133,2,59,40,run_session_logreg__Assembly001
1,2024-12-02_16-09,Outcome,outcome_binary,R1_entry_interval,105,2,45,40,run_session_logreg__Assembly001
2,2024-12-03_16-23,Outcome,outcome_binary,R1_entry_interval,104,2,47,40,run_session_logreg__Assembly001
3,2024-12-04_18-06,Outcome,outcome_binary,R1_entry_interval,120,2,55,40,run_session_logreg__Assembly001
4,2024-12-06_16-49,Outcome,outcome_binary,R1_entry_interval,95,2,43,40,run_session_logreg__Assembly001


In [34]:
def _coerce_bool_flag(raw) -> bool:
    if pd.isna(raw):
        return False
    if isinstance(raw, (bool, np.bool_)):
        return bool(raw)
    if isinstance(raw, (int, np.integer, float, np.floating)):
        return bool(raw)
    return str(raw).strip().lower() in {'true', '1', 'yes', 'y', 't'}


def make_chance_neutral_colorscale(
    chance_level: float = 0.5,
    zmin: float = 0.0,
    zmax: float = 1.0,
    neutral_half_width: float = 0.08,
):
    if zmax <= zmin:
        raise ValueError('zmax must be larger than zmin.')
    center = float(np.clip((chance_level - zmin) / (zmax - zmin), 0.0, 1.0))
    half_width = max(0.0, neutral_half_width / (zmax - zmin))
    lower = float(np.clip(center - half_width, 0.0, 1.0))
    upper = float(np.clip(center + half_width, 0.0, 1.0))
    low_mid = float(np.clip(lower - 0.14, 0.0, 1.0))
    high_mid = float(np.clip(upper + 0.14, 0.0, 1.0))
    return [
        [0.0, '#2b5876'],
        [low_mid, '#7aa6c2'],
        [lower, '#d0d0d0'],
        [center, '#ffffff'],
        [upper, '#d0d0d0'],
        [high_mid, '#f0a36b'],
        [1.0, '#b24745'],
    ]


def plot_session_interval_heatmaps(
    results_df: pd.DataFrame,
    value_col: str = 'balanced_accuracy',
    title_prefix: str = 'Session-level decoding',
    extra_cols=None,
    ncols: int = 3,
    colorscale: str = 'RdBu_r',
    zmid: float = 0.5,
    significance_col=None,
    significance_label: str = 'shuffle significant (p<0.05)',
    min_trials=None,
    significance_marker_size: float = 7.0,
    interval_names=None,
):
    if extra_cols is None:
        extra_cols = []

    valid = results_df[results_df[value_col].notna()].copy()
    if min_trials is not None and 'n_trials' in valid.columns:
        valid = valid[pd.to_numeric(valid['n_trials'], errors='coerce') >= float(min_trials)].copy()
    if valid.empty:
        trial_suffix = f' after filtering to n_trials >= {min_trials}' if min_trials is not None else ''
        print(f'No rows with {value_col}{trial_suffix}.')
        return

    separator_date = datetime.strptime('2025-01-23', '%Y-%m-%d')

    for run_name, run_df in valid.groupby('run_name', sort=False):
        run_df, intervals = _filter_interval_rows(run_df, interval_names)
        if run_df.empty or not intervals:
            continue
        targets = _target_order(run_df['target_name'].astype(str))
        all_sessions = sorted(run_df['session_id'].astype(str).unique().tolist(), key=_session_sort_key)
        bounded_value_range = value_col in {'accuracy', 'balanced_accuracy', 'bootstrap_mean'}

        nrows = int(np.ceil(len(intervals) / ncols))
        fig = make_subplots(
            rows=nrows,
            cols=ncols,
            subplot_titles=intervals,
            horizontal_spacing=0.08,
            vertical_spacing=0.18 if nrows > 1 else 0.10,
        )

        target_to_i = {target: idx for idx, target in enumerate(targets)}

        for interval_idx, interval_name in enumerate(intervals):
            row = (interval_idx // ncols) + 1
            col = (interval_idx % ncols) + 1
            sub = run_df[run_df['interval_name'].astype(str) == str(interval_name)].copy()
            sessions_present = set(sub['session_id'].astype(str))
            sessions = [session for session in all_sessions if session in sessions_present]
            if not sessions:
                continue
            session_tick_text = [session[:10] if _parse_session_date_token(session) else session for session in sessions]
            session_to_j = {session: idx for idx, session in enumerate(sessions)}
            separator_idx = next(
                (idx for idx, session in enumerate(sessions) if _parse_session_date_token(session) == separator_date),
                None,
            )

            z = np.full((len(targets), len(sessions)), np.nan, dtype=float)
            custom = np.full((len(targets), len(sessions), len(extra_cols)), np.nan, dtype=float)
            significance = np.zeros((len(targets), len(sessions)), dtype=bool) if significance_col else None
            for rec in sub.itertuples(index=False):
                i = target_to_i[str(rec.target_name)]
                j = session_to_j[str(rec.session_id)]
                value = getattr(rec, value_col)
                z[i, j] = float(value) if pd.notna(value) else np.nan
                for extra_idx, extra_col in enumerate(extra_cols):
                    raw = getattr(rec, extra_col) if hasattr(rec, extra_col) else np.nan
                    custom[i, j, extra_idx] = float(raw) if pd.notna(raw) else np.nan
                if significance is not None and hasattr(rec, significance_col):
                    significance[i, j] = _coerce_bool_flag(getattr(rec, significance_col))

            hover_lines = [
                f'{value_col}=%{{z:.3f}}',
            ]
            for extra_idx, extra_col in enumerate(extra_cols):
                hover_lines.append(f'{extra_col}=%{{customdata[{extra_idx}]:.3f}}')
            hovertemplate = 'interval=' + str(interval_name) + '<br>target=%{y}<br>session=%{x}<br>' + '<br>'.join(hover_lines) + '<extra></extra>'

            fig.add_trace(
                go.Heatmap(
                    x=sessions,
                    y=targets,
                    z=z,
                    customdata=custom,
                    coloraxis='coloraxis',
                    hovertemplate=hovertemplate,
                ),
                row=row,
                col=col,
            )

            if separator_idx is not None:
                axis_suffix = '' if interval_idx == 0 else str(interval_idx + 1)
                separator_x = separator_idx - 0.5
                fig.add_shape(
                    type='line',
                    x0=separator_x,
                    x1=separator_x,
                    y0=0,
                    y1=1,
                    xref=f'x{axis_suffix}',
                    yref=f'y{axis_suffix} domain',
                    line={'color': 'red', 'width': 2},
                )

            if significance is not None:
                marker_x = []
                marker_y = []
                for target_idx, target_name in enumerate(targets):
                    for session_idx, session_name in enumerate(sessions):
                        if significance[target_idx, session_idx] and np.isfinite(z[target_idx, session_idx]):
                            marker_x.append(session_name)
                            marker_y.append(target_name)
                if marker_x:
                    fig.add_trace(
                        go.Scatter(
                            x=marker_x,
                            y=marker_y,
                            mode='markers',
                            marker={
                                'size': significance_marker_size,
                                'color': 'rgba(255, 255, 255, 0.95)',
                                'line': {'color': 'black', 'width': 1.25},
                            },
                            name=significance_label,
                            legendgroup='shuffle_significance',
                            showlegend=interval_idx == 0,
                            hovertemplate=significance_label + '<br>target=%{y}<br>session=%{x}<extra></extra>',
                        ),
                        row=row,
                        col=col,
                    )
            fig.update_xaxes(
                tickmode='array',
                tickvals=sessions,
                ticktext=session_tick_text,
                tickangle=45,
                title_text='session',
                row=row,
                col=col,
            )
            fig.update_yaxes(title_text='target', row=row, col=col)

        coloraxis = {
            'colorscale': colorscale,
            'colorbar': {'title': value_col},
        }
        if bounded_value_range:
            coloraxis['cmin'] = 0.0
            coloraxis['cmax'] = 1.0
        if zmid is not None:
            coloraxis['cmid'] = zmid

        fig.update_layout(
            template='plotly_white',
            title=f'{title_prefix} | run={run_name}',
            coloraxis=coloraxis,
            height=max(220, int(300 * nrows)),
            width=max(1100, int(420 * ncols)),
            margin={'l': 80, 'r': 40, 't': 90, 'b': 100},
        )
        fig.show()


### Session Progression Heatmaps

Each subplot is one interval. Columns are sessions with at least 10 trials for that interval, rows are decoded targets, and hover contains bootstrap and shuffle diagnostics. White marker dots indicate shuffle-significant cells (`significant_shuffle_0_05`).


In [35]:
plot_session_interval_heatmaps(
    session_results_df,
    value_col='balanced_accuracy',
    title_prefix='Session-level balanced accuracy',
    extra_cols=['bootstrap_ci_low', 'bootstrap_ci_high', 'p_bootstrap_above_chance', 'p_shuffle', 'p_shuffle_fdr_bh', 'n_trials', 'n_splits'],
    ncols=3,
    colorscale=make_chance_neutral_colorscale(neutral_half_width=0.08),
    zmid=0.5,
    significance_col='significant_shuffle_fdr_bh_0_05',
    significance_label='shuffle significant (BH-FDR q<0.05)',
    min_trials=10,
    significance_marker_size=4,
    interval_names=BALANCED_ACCURACY_INTERVALS,
)


### Session-Level Shuffle Effect


In [36]:
# plot_session_interval_heatmaps(
#     session_results_df,
#     value_col='shuffle_effect',
#     title_prefix='Session-level shuffle effect',
#     extra_cols=['shuffle_null_mean', 'p_shuffle', 'bootstrap_ci_low', 'bootstrap_ci_high'],
#     ncols=3,
#     colorscale='RdBu_r',
#     zmid=0.0,
# )
